# Group 2 - House Sales: Predicting the Sale Price

**Team members:** Karthikeyan Saravanan, Emmanuel Marquez, Bars Babila, Samiha Abeda Nur

**Course:** AI Methods I – Day 2: Supervised learning (IAAC MaAI 2026/27)

**Model on Hugging Face:** https://huggingface.co/iaac-maai/day2-team2-house-price

**Day 1 repository:** https://github.com/iaac-maai/2-day1-assignment

In this notebook we continue our Day 1 work on house sales in King County, Washington, USA. We use the cleaned dataset from Day 1 to train a model that predicts the sale price of a house from its physical characteristics and its location.

# 0. Introduction and Problem Definition

## Use case

Before starting, we answered three questions:

| Question | Our answer |
|---|---|
| **Who would use this model, and for what?** | An architect or a small developer in King County doing a feasibility study. They compare design options for a house (size, bedrooms, bathrooms, floors, construction quality) and want a quick estimate of the market value of each option. |
| **Which mistake is more expensive?** | Over-estimating the price. If the model says a design will sell for 700,000 USD and it sells for 600,000 USD, the developer may have paid too much for the land or the construction. |
| **What would the user know at prediction time?** | The design of the house (living area, bedrooms, bathrooms, floors, basement, grade, condition, year built and renovated) and the plot (lot area, ZIP code and coordinates). They would not know the sale price or the sale date. |

In [6]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

In [7]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib   # to save and load the trained model

### Folder Directory

In [8]:
# --- Folders --------------------------------------------------------------
DATA_DIR = Path("..") / "data"      # the clean CSV file from Day 1 is here
FIG_DIR = Path("..") / "figures"    # exported plots go here
MODEL_DIR = Path("..") / "models"   # the trained model is saved here
for folder in [DATA_DIR, FIG_DIR, MODEL_DIR]:
    folder.mkdir(exist_ok=True)

# --- pandas display options -------------------------------------------------
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

# --- plot style -------------------------------------------------------------
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

# Seed for anything random (split, cross-validation, models),
# so we always get the same results.
RANDOM_STATE = 42


def save_fig(name):
    # Save the current figure as a PNG inside FIG_DIR.
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figure saved to {path}")

## Load the Clean Dataset from Day 1

We start from the clean dataset we produced on Day 1 (`kc_houses_data_clean.csv`). On Day 1 we:

- converted the `date` column from a string to a datetime format,
- dropped the `waterfront` and `view` columns, because their values are heavily concentrated at zero,
- removed the records below the 1st and above the 99th percentile of price, living area and bedroom count (910 records removed).

In [9]:
clean_path = DATA_DIR / "kc_houses_data_clean.csv"

if not clean_path.exists():
    raise FileNotFoundError(
        f"{clean_path} not found. Run the Day 1 notebook first and copy "
        "data/kc_houses_data_clean.csv into this repository's data/ folder."
    )

houses = pd.read_csv(clean_path, parse_dates=["date"])
print(f"Loaded {houses.shape[0]:,} rows and {houses.shape[1]} columns")
houses.head()

Loaded 20,703 rows and 19 columns


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,2014-10-13,"221,900.00",3,1.00,1180,5650,1.00,3,7,1180,0,1955,0,98178,47.51,-122.26,1340,5650
1,6414100192,2014-12-09,"538,000.00",3,2.25,2570,7242,2.00,3,7,2170,400,1951,1991,98125,47.72,-122.32,1690,7639
2,5631500400,2015-02-25,"180,000.00",2,1.00,770,10000,1.00,3,6,770,0,1933,0,98028,47.74,-122.23,2720,8062
3,2487200875,2014-12-09,"604,000.00",4,3.00,1960,5000,1.00,5,7,1050,910,1965,0,98136,47.52,-122.39,1360,5000
4,1954400510,2015-02-18,"510,000.00",3,2.00,1680,8080,1.00,3,8,1680,0,1987,0,98074,47.62,-122.05,1800,7503


In [10]:
houses.info()

<class 'pandas.DataFrame'>
RangeIndex: 20703 entries, 0 to 20702
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             20703 non-null  int64         
 1   date           20703 non-null  datetime64[us]
 2   price          20703 non-null  float64       
 3   bedrooms       20703 non-null  int64         
 4   bathrooms      20703 non-null  float64       
 5   sqft_living    20703 non-null  int64         
 6   sqft_lot       20703 non-null  int64         
 7   floors         20703 non-null  float64       
 8   condition      20703 non-null  int64         
 9   grade          20703 non-null  int64         
 10  sqft_above     20703 non-null  int64         
 11  sqft_basement  20703 non-null  int64         
 12  yr_built       20703 non-null  int64         
 13  yr_renovated   20703 non-null  int64         
 14  zipcode        20703 non-null  int64         
 15  lat            20703 non-null 

## Problem Type, Target and Main Metric

- **Problem type:** regression, because the target is a number.
- **Target:** `price`, the sale price of the house in USD.
- **Main metric:** MAE (mean absolute error) in USD. It tells the user directly how many dollars the estimate is off on average, and it is less affected by a few very expensive houses than RMSE.

We also report RMSE, R² and MAPE (the error as a percentage of the price). Because over-estimating is the more expensive mistake for our user, we will also check how often the model over-estimates the price in the final evaluation.